In [0]:
%run ./config

In [0]:
df=spark.read.parquet(gold+'/race_results')


In [0]:
standings_df= df.groupBy('race_year', 'driver_name', 'team', 'driver_nationality')\
                .agg(sum('points').alias('total_points'), 
                count(when(col('position')==1, True)).alias('wins'))\
                .orderBy(desc('total_points'), desc('wins'))

In [0]:
rank_spec=Window.partitionBy('race_year').orderBy(desc('total_points'), desc('wins'))
standings_df= standings_df.withColumn('rank', dense_rank().over(rank_spec))

In [0]:
# standings_df.filter('race_year=2019').display()
standings_df.write.mode('overwrite').partitionBy('race_year').parquet(gold+'/driver_standings')